# Notebook 05: Anatomical Body-Region Analysis & Topology Verification

**Objective**: Audit and visualize the anatomical grouping of on-body sensor channels and construct the baseline skeletal graph prior.

### Grounded Documentation:
- **Trunk (19 ch)**: Back & Hip custom 3D accelerometers, Back IMU.
- **Right Arm (38 ch)**: Right Upper Arm, Lower Arm, Wrist, and Hand IMUs.
- **Left Arm (38 ch)**: Left Upper Arm, Lower Arm, Wrist, and Hand IMUs.
- **Right Leg (22 ch)**: Right Knee accelerometers and Right Shoe IMU.
- **Left Leg (16 ch)**: Left Shoe IMU.

**Compute Environment**: CPU Safe.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("Cloning Opportunity_HAR from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

import src.api as api
from src.features.body_regions import (
    BODY_REGIONS,
    ANATOMICAL_SENSOR_COLUMNS,
    get_fixed_adjacency_matrix,
    plot_body_region_graph,
)
env_info = api.setup_project()
print(f"Project Initialized on: {env_info['device_name']} (CUDA: {env_info['cuda_available']})")

## 1. Audit Sensor Channel Completeness & Mutuality

In [ ]:
total_channels = 0
all_mapped_cols = []

print("Anatomical Body-Region Mapping Table:")
print("-" * 50)
for region in BODY_REGIONS:
    cols = ANATOMICAL_SENSOR_COLUMNS[region]
    total_channels += len(cols)
    all_mapped_cols.extend(cols)
    print(f"  {region.replace('_', ' ').title():12s}: {len(cols):2d} channels  (Cols: {cols[:3]} ... {cols[-2:]})")
print("-" * 50)
print(f"Total On-Body Channels: {total_channels}")

# Verification checks
assert total_channels == 133, f"Expected 133 on-body channels, got {total_channels}"
assert len(set(all_mapped_cols)) == 133, "Found duplicate channel assignments across body regions!"
assert min(all_mapped_cols) == 1 and max(all_mapped_cols) == 133, "Channel index out of on-body range (1-133)"
print("\n[OK] Verified: All 133 on-body channels are mapped exactly once with zero overlap!")

## 2. Visualize Fixed Anatomical Skeletal Topology Graph

In [ ]:
import matplotlib.pyplot as plt
fig = plot_body_region_graph()
plt.show()

## 3. Fixed Adjacency Matrix & Row-Normalized Graph Prior

In [ ]:
import pandas as pd
raw_adj = get_fixed_adjacency_matrix()
df_adj = pd.DataFrame(raw_adj, index=BODY_REGIONS, columns=BODY_REGIONS)

print("Fixed Skeletal Binary Adjacency Matrix A (with self-loops):")
display(df_adj)

# Row-normalized transition matrix
norm_adj = raw_adj / raw_adj.sum(axis=1, keepdims=True)
df_norm = pd.DataFrame(norm_adj, index=BODY_REGIONS, columns=BODY_REGIONS)
print("\nRow-Normalized Adjacency Matrix (Incoming Diffusion Weights):")
display(df_norm.round(3))